# 05. First-Order Logic

Notebook 03 menunjukkan batas propositional logic: nama seperti `W12` dan
`W13` diperlakukan sebagai atom terpisah. **First-order logic** (FOL) memberi
struktur dengan menunjukkan object yang dibicarakan serta property, relation,
dan function yang berlaku padanya.

Notebook ini memakai domain kampus sebagai contoh utama. Kasus Colonel West
diberikan di akhir sebagai latihan transfer dan menjadi knowledge base untuk
inference pada Notebook 06.

Setelah menyelesaikan notebook ini, Anda diharapkan mampu:

1. menjelaskan mengapa FOL diperlukan;
2. membedakan isi dunia dari simbol yang menuliskannya;
3. membedakan term, formula atomik, formula majemuk, formula terbuka, dan
   sentence;
4. menjelaskan truth sebuah formula terhadap domain dan interpretation;
5. menggunakan quantifier, scope, serta free/bound variable;
6. menerjemahkan natural language ke FOL; dan
7. membedakan notasi FOL formal dari encoding `Expr` dan `FolKB`.


## Setup

Jalankan sel di bawah ini sekali di awal, sebelum sel lain.

Sel ini memasang dependensi yang diperlukan, mencari folder berisi `logic.py`
dan `utils.py`, lalu mengimpornya. Jika notebook dibuka lewat Google Colab,
repo akan di-clone otomatis.

Environment siap jika baris terakhir mencetak
`Check       : tt_entails(P & Q, Q) = True`.


In [3]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules


def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None


ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )
if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings

import pandas as pd

# qpsolvers is only used by the SVM code, which this module never touches.
warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))


RuntimeError: Environment folder not found. Make sure this notebook is opened from inside the Modul-Praktikum-KK-RKA-25 repository.

---
# 5.1 Dari Dunia ke Ekspresi FOL

## Mengapa First-Order Logic?

Propositional logic dapat memberi nama pada fact:

```text
StudentAni
StudentBudi
TakesAniAI
TakesBudiAI
```

Namun setiap nama dianggap satu atom terpisah. Sistem tidak melihat bahwa dua
nama pertama memakai konsep mahasiswa yang sama atau bahwa dua nama terakhir
memakai relation mengambil yang sama.

FOL membuat struktur itu terlihat:

```text
Student(Ani)
Student(Budi)
Takes(Ani, AI)
Takes(Budi, AI)
```

- `Ani`, `Budi`, dan `AI` menamai object;
- `Student` menyatakan property;
- `Takes` menyatakan relation;
- argument menunjukkan object yang sedang dibicarakan.

Dengan struktur tersebut, satu rule dapat berlaku untuk banyak object.

## Contoh penerapan

Sebelum menjalankan sel, prediksi `.op` dan `.args` dari bentuk propositional
dan FOL berikut.


In [ ]:
# Propositional symbol: nama atomik tanpa struktur internal.
student_ani_prop = Symbol('StudentAni')

# FOL atomic formula: predicate Student diterapkan ke constant Ani.
student_ani_fol = expr('Student(Ani)')

print("Propositional:", student_ani_prop)
print("  op   =", student_ani_prop.op)
print("  args =", student_ani_prop.args)
print("FOL:", student_ani_fol)
print("  op   =", student_ani_fol.op)
print("  args =", student_ani_fol.args)


Propositional: StudentAni
  op   = StudentAni
  args = ()
FOL: Student(Ani)
  op   = Student
  args = (Ani,)


`StudentAni` tidak memiliki argumen. `Student(Ani)` memiliki predicate
`Student` dan satu argumen `Ani`. Struktur ini memungkinkan satu rule berlaku
untuk banyak object.


---
## 5.1.1 Dari isi dunia ke simbol FOL

Jangan mencampur **isi dunia** dengan **cara menuliskannya**.

| Isi dunia | Symbol FOL | Contoh |
|---|---|---|
| object tertentu | **constant**: nama object | `Ani`, `AI` |
| object belum ditentukan | **variable**: placeholder | `x`, `y` |
| property/relation | **predicate symbol**: nama property/relation | `Student`, `Takes` |
| pemetaan ke object lain | **function symbol**: nama pemetaan | `AdvisorOf` |

Contoh:

```text
Student(Ani)
```

- `Ani` adalah constant yang menamai object Ani;
- `Student` adalah predicate symbol yang menamai property mahasiswa;
- `Student(Ani)` adalah formula yang dapat benar atau salah.

Bandingkan:

```text
AdvisorOf(Ani)
```

- `AdvisorOf` adalah function symbol;
- `AdvisorOf(Ani)` menunjuk object dosen wali Ani;
- hasilnya term, bukan pernyataan benar atau salah.

Cara cepat membedakan:

```text
Menghasilkan object    → function symbol
Membentuk benar/salah  → predicate symbol
```

Jumlah argument disebut **arity**. `Student` ber-arity 1, `Takes` ber-arity 2,
dan `Sells` ber-arity 3.

## 5.1.2 Lapisan pembentukan ekspresi

```text
Ani / x
    ↓ constant atau variable adalah term
AdvisorOf(Ani)
    ↓ function application juga term
Student(Ani)
    ↓ predicate application membentuk formula atomik
Student(x) ∧ Takes(x, AI)
    ↓ connective membentuk formula majemuk
∀x(Student(x) ⇒ Smart(x))
    ↓ semua variable terikat; menjadi sentence
```

Gunakan dua tes:

- **Menunjuk object?** Ekspresi itu term.
- **Dapat benar atau salah?** Ekspresi itu formula.

Istilah penting:

| Istilah | Makna | Contoh |
|---|---|---|
| Term | menunjuk object | `AdvisorOf(Ani)` |
| Formula atomik | pernyataan sederhana | `Student(Ani)` |
| Formula majemuk | gabungan formula | `Student(x) ∧ Smart(x)` |
| Formula terbuka | masih memiliki free variable | `Student(x)` |
| Sentence | tidak memiliki free variable | `∀x Student(x)` |

Equality juga membentuk formula atomik. `AdvisorOf(Ani) = DrRina` menyatakan
bahwa kedua term menunjuk object yang sama.

## 5.1.3 Panduan notasi

| Makna | FOL formal | `Expr`/AIMA |
|---|---|---|
| negasi | `¬P` | `~P` |
| dan | `P ∧ Q` | `P & Q` |
| atau | `P ∨ Q` | `P | Q` |
| implication | `P ⇒ Q` | `P ==> Q` |
| biconditional | `P ⇔ Q` | `P <=> Q` |
| universal | `∀x P(x)` | tidak tersedia eksplisit |
| existential | `∃x P(x)` | tidak tersedia eksplisit |

Saat menjelaskan makna, gunakan notasi formal. Saat menjalankan Python, gunakan
notasi AIMA. Jangan mencampur keduanya dalam satu formula.

## 5.1.4 Konvensi library

`logic.py` menganggap nama berawalan huruf kecil sebagai variable (`x`, `y`) dan
nama berawalan huruf besar sebagai constant, predicate, atau function
(`Ani`, `Student`, `AdvisorOf`). Ini konvensi AIMA, bukan aturan universal teori
FOL.


## Contoh penerapan

Bandingkan term, formula atomik, formula terbuka, dan rule. Perhatikan bahwa
term dapat menjadi argument di dalam predicate.


In [ ]:
x = expr('x')
ani = expr('Ani')
advisor_term = expr('AdvisorOf(Ani)')
lecturer_atom = expr('Lecturer(AdvisorOf(Ani))')
takes_open = expr('Takes(x, AI)')
eligible_rule = expr('(Student(x) & Takes(x, AI)) ==> Eligible(x, AILab)')

examples = [x, ani, advisor_term, lecturer_atom, takes_open, eligible_rule]
for item in examples:
    print(f"{str(item):55} op={item.op!r:12} args={item.args}")

print()
print("Variable dalam Takes(x, AI):", variables(takes_open))
print("Variable dalam rule          :", variables(eligible_rule))
print("'x' variable?                :", is_var_symbol('x'))
print("'Ani' variable?              :", is_var_symbol('Ani'))


x                                                       op='x'          args=()
Ani                                                     op='Ani'        args=()
AdvisorOf(Ani)                                          op='AdvisorOf'  args=(Ani,)
Lecturer(AdvisorOf(Ani))                                op='Lecturer'   args=(AdvisorOf(Ani),)
Takes(x, AI)                                            op='Takes'      args=(x, AI)
((Student(x) & Takes(x, AI)) ==> Eligible(x, AILab))    op='==>'        args=((Student(x) & Takes(x, AI)), Eligible(x, AILab))

Variable dalam Takes(x, AI): {x}
Variable dalam rule          : {x}
'x' variable?                : True
'Ani' variable?              : False


`Expr` menyimpan syntax sebagai tree: `.op` adalah akar dan `.args` adalah
cabangnya. Struktur ini belum menentukan apakah formula benar. Kebenaran formula
ditentukan oleh semantics.


---
# 5.2 Semantics: Kapan Formula Benar?

## Penjelasan

**Syntax** membahas bentuk ekspresi. **Semantics** membahas arti ekspresi dan
kapan ekspresi itu benar pada suatu model.

Interpretation sederhana memerlukan:

1. **domain** $D$: seluruh object yang sedang dibicarakan;
2. arti constant: object yang ditunjuk;
3. arti predicate: object atau tuple yang termasuk property/relation;
4. arti function: object keluaran untuk setiap input; dan
5. assignment untuk free variable.

Contoh:

```text
D       = {Ani, Budi, Kursi}
Student = {Ani, Budi}
Smart   = {Ani}
```

Maka:

- `Student(Ani)` benar;
- `Student(Kursi)` salah;
- `Smart(Budi)` salah;
- `Student(x)` belum memiliki truth value sampai `x` mendapat assignment.

Quantifier menjangkau **seluruh domain**, termasuk `Kursi`. Predicate
`Student(x)` dipakai untuk membatasi object yang dimaksud.

## Contoh penerapan

Kode berikut bukan parser FOL. Kode hanya memperlihatkan semantics quantifier
pada domain berhingga.


In [ ]:
domain = {'Ani', 'Budi', 'Kursi'}
student_set = {'Ani', 'Budi'}
smart_set = {'Ani'}

student_ani = 'Ani' in student_set
student_chair = 'Kursi' in student_set
all_students_smart = all(
    (obj not in student_set) or (obj in smart_set)
    for obj in domain
)
some_student_smart = any(
    (obj in student_set) and (obj in smart_set)
    for obj in domain
)

counterexamples = sorted(
    obj for obj in domain
    if obj in student_set and obj not in smart_set
)
witnesses = sorted(
    obj for obj in domain
    if obj in student_set and obj in smart_set
)

print("Student(Ani)                         =", student_ani)
print("Student(Kursi)                       =", student_chair)
print("Semua mahasiswa pintar              =", all_students_smart)
print("  counterexample                     =", counterexamples)
print("Ada mahasiswa yang pintar           =", some_student_smart)
print("  witness                            =", witnesses)


Student(Ani)                         = True
Student(Kursi)                       = False
Semua mahasiswa pintar              = False
  counterexample                     = ['Budi']
Ada mahasiswa yang pintar           = True
  witness                            = ['Ani']


Satu **counterexample** cukup untuk menggugurkan universal statement. Satu
**witness** cukup untuk membuktikan existential statement pada model berhingga
ini. Budi menggugurkan “semua mahasiswa pintar”; Ani menjadi witness untuk “ada
mahasiswa pintar”.


---
# 5.3 Quantifier, Scope, dan Urutan

## 5.3.1 Universal dan existential quantifier

**Universal quantifier**:

$$\forall x\,P(x)$$

benar jika $P(x)$ benar untuk setiap object di domain.

**Existential quantifier**:

$$\exists x\,P(x)$$

benar jika ada sedikitnya satu object di domain yang membuat $P(x)$ benar.

Dua pola translasi yang sering dipakai:

| Kalimat | Formula |
|---|---|
| Semua A adalah B | $\forall x\,(A(x)\Rightarrow B(x))$ |
| Ada A yang B | $\exists x\,(A(x)\land B(x))$ |

Ini **pola translasi**, bukan aturan bahwa $\forall$ selalu wajib memakai
$\Rightarrow$ atau $\exists$ selalu wajib memakai $\land$. Formula
$\forall x(P(x)\land Q(x))$ sah jika memang bermaksud menyatakan bahwa setiap
object memiliki kedua property.

### Dua kesalahan makna yang umum

$$\forall x\,(Student(x)\land Smart(x))$$

menyatakan bahwa **setiap object di domain** adalah mahasiswa dan pintar,
termasuk kursi. Ini bukan “semua mahasiswa pintar”.

$$\exists x\,(Student(x)\Rightarrow Smart(x))$$

mudah menjadi benar hanya karena ada satu object yang bukan mahasiswa. Jika
`x = Kursi`, premise `Student(Kursi)` salah sehingga implication benar secara
vacuous.

## Contoh penerapan


In [ ]:
correct_universal = all(
    (obj not in student_set) or (obj in smart_set)
    for obj in domain
)
wrong_universal = all(
    (obj in student_set) and (obj in smart_set)
    for obj in domain
)
correct_existential = any(
    (obj in student_set) and (obj in smart_set)
    for obj in domain
)
wrong_existential = any(
    (obj not in student_set) or (obj in smart_set)
    for obj in domain
)
vacuous_witnesses = sorted(obj for obj in domain if obj not in student_set)

print("∀x (Student(x) ⇒ Smart(x)) =", correct_universal)
print("∀x (Student(x) ∧ Smart(x)) =", wrong_universal)
print("∃x (Student(x) ∧ Smart(x)) =", correct_existential)
print("∃x (Student(x) ⇒ Smart(x)) =", wrong_existential)
print("Non-mahasiswa yang membuat implication vacuously true:", vacuous_witnesses)


∀x (Student(x) ⇒ Smart(x)) = False
∀x (Student(x) ∧ Smart(x)) = False
∃x (Student(x) ∧ Smart(x)) = True
∃x (Student(x) ⇒ Smart(x)) = True
Non-mahasiswa yang membuat implication vacuously true: ['Kursi']


## 5.3.2 Free variable, bound variable, dan scope

Pada `Takes(x, AI)`, `x` adalah **free variable**. Nilai kebenarannya bergantung
pada assignment untuk `x`.

Pada $\forall x\,Takes(x,AI)$, kemunculan `x` di dalam scope $\forall x$ adalah
**bound variable**. Formula tersebut menjadi sentence.

Tanda kurung menentukan scope:

$$\forall x\,(Student(x)\Rightarrow Smart(x))$$

berbeda dari

$$(\forall x\,Student(x))\Rightarrow Smart(x)$$

Pada formula kedua, `x` dalam `Smart(x)` berada di luar scope quantifier dan
masih free. Selalu tulis tanda kurung eksplisit saat menerjemahkan kalimat.

Nama bound variable boleh diganti tanpa mengubah makna:

$$\forall x\,Student(x)\equiv\forall y\,Student(y)$$

selama penggantian tidak membuat variable tertangkap oleh quantifier lain.

## 5.3.3 Urutan quantifier

Urutan quantifier berbeda jenis mengubah makna:

$$\forall x\,(Student(x)\Rightarrow\exists y\,
(Course(y)\land Takes(x,y)))$$

“Setiap mahasiswa mengambil sedikitnya satu mata kuliah.” Mata kuliah untuk tiap
mahasiswa boleh berbeda.

$$\exists y\,(Course(y)\land\forall x\,
(Student(x)\Rightarrow Takes(x,y)))$$

“Ada satu mata kuliah yang diambil semua mahasiswa.” Mata kuliahnya harus sama.

Quantifier sejenis dapat ditukar urutannya, tetapi quantifier berbeda jenis
umumnya tidak:

$$\forall x\forall y\,P(x,y)\equiv\forall y\forall x\,P(x,y)$$

$$\exists x\forall y\,P(x,y)\not\equiv\forall y\exists x\,P(x,y)$$

## Contoh penerapan


In [ ]:
students = {'Ani', 'Budi'}
courses = {'AI', 'BasisData'}
takes = {('Ani', 'AI'), ('Budi', 'BasisData')}

every_student_takes_some_course = all(
    any((student, course) in takes for course in courses)
    for student in students
)
one_course_taken_by_every_student = any(
    all((student, course) in takes for student in students)
    for course in courses
)

print("∀ mahasiswa ∃ mata kuliah =", every_student_takes_some_course)
print("∃ mata kuliah ∀ mahasiswa =", one_course_taken_by_every_student)


∀ mahasiswa ∃ mata kuliah = True
∃ mata kuliah ∀ mahasiswa = False


Kedua mahasiswa mengambil mata kuliah, sehingga formula pertama benar. Tidak
ada satu mata kuliah yang diambil keduanya, sehingga formula kedua salah.

## 5.3.4 Negasi quantifier

Quantifier memiliki duality:

$$\forall x\,P(x)\equiv\neg\exists x\,\neg P(x)$$

$$\exists x\,P(x)\equiv\neg\forall x\,\neg P(x)$$

Contoh:

- “Tidak ada mahasiswa yang terlambat”:
  $\neg\exists x(Student(x)\land Late(x))$.
- Bentuk ekuivalen:
  $\forall x(Student(x)\Rightarrow\neg Late(x))$.
- “Tidak semua mahasiswa lulus”:
  $\neg\forall x(Student(x)\Rightarrow Passed(x))$.
- Bentuk ekuivalen:
  $\exists x(Student(x)\land\neg Passed(x))$.


---
# 5.4 Menerjemahkan Natural Language ke FOL

## Penjelasan

Gunakan enam langkah. Jangan langsung mengganti setiap kata dengan simbol.

1. **Tentukan domain.** Object apa saja yang dapat dirujuk variable?
2. **Tentukan vocabulary.** Daftar constant, predicate, function, dan arity.
3. **Tulis formula atomik.** Pecah kalimat menjadi pernyataan sederhana.
4. **Gabungkan syarat.** Gunakan connective dan tanda kurung.
5. **Tambahkan quantifier.** Pastikan scope dan urutannya benar.
6. **Baca balik dan uji.** Cari counterexample atau witness untuk memeriksa
   apakah maknanya sesuai.

### Tangga translasi

Anggap domain memuat seluruh object kampus.

| Natural language | FOL formal |
|---|---|
| Ani mahasiswa | $Student(Ani)$ |
| Ani mengambil AI | $Takes(Ani,AI)$ |
| Semua mahasiswa pintar | $\forall x(Student(x)\Rightarrow Smart(x))$ |
| Ada mahasiswa pintar | $\exists x(Student(x)\land Smart(x))$ |
| Tidak ada mahasiswa terlambat | $\neg\exists x(Student(x)\land Late(x))$ |
| Setiap mahasiswa mengambil suatu mata kuliah | $\forall x(Student(x)\Rightarrow\exists y(Course(y)\land Takes(x,y)))$ |

### Function atau predicate?

- `AdvisorOf(Ani)` adalah term yang menunjuk object.
- `Advises(DrRina, Ani)` adalah formula yang menyatakan relation.

Keduanya dapat merepresentasikan informasi terkait, tetapi jenis ekspresinya
berbeda. Jangan menanyakan apakah term `AdvisorOf(Ani)` benar atau salah.


---
# 5.5 FOL Formal vs `Expr` dan `FolKB`

## Penjelasan

FOL formal mendukung formula umum serta quantifier `∀` dan `∃`. Library
praktikum memakai subset yang lebih sempit.

`FolKB` menerima first-order definite clauses:

```text
Fact
A & B & C ==> D
```

Contoh formal:

$$\forall x\,((Student(x)\land Takes(x,AI))
\Rightarrow Eligible(x,AILab))$$

Encoding `FolKB`:

```text
(Student(x) & Takes(x, AI)) ==> Eligible(x, AILab)
```

Variable rule dianggap universal secara implisit. Batas ini milik tool, bukan
batas FOL sebagai bahasa.

## Contoh penerapan


In [ ]:
rule = expr('(Student(x) & Takes(x, AI)) ==> Eligible(x, AILab)')
non_definite = expr('Student(Ani) | Lecturer(Ani)')

print("Rule:", rule)
print("Variable yang dianggap universal:", variables(rule))
print("Rule definite clause?              ", is_definite_clause(rule))
print("Disjunction definite clause?       ", is_definite_clause(non_definite))


Rule: ((Student(x) & Takes(x, AI)) ==> Eligible(x, AILab))
Variable yang dianggap universal: {x}
Rule definite clause?               True
Disjunction definite clause?        False


## Existential statement dalam KB executable

Formula formal:

$$\exists x\,(Student(x)\land Takes(x,AI))$$

`FolKB` tidak dapat menyimpan quantifier itu langsung. Untuk encoding
executable sederhana, perkenalkan constant baru sebagai witness:

```text
Student(S1)
Takes(S1, AI)
```

`S1` harus **fresh**, yaitu belum memiliki arti lain dalam KB. Formula formal
menyatakan bahwa object tersebut ada; encoding executable memberi nama pada
object yang diketahui ada.

Selalu tampilkan dua tahap:

1. formula FOL formal;
2. encoding definite-clause yang dapat dijalankan.


In [ ]:
# Encoding executable untuk satu witness existential yang fresh.
existential_witness_facts = [
    expr('Student(S1)'),
    expr('Takes(S1, AI)'),
]

for fact in existential_witness_facts:
    print(fact, "-> definite clause?", is_definite_clause(fact))


Student(S1) -> definite clause? True
Takes(S1, AI) -> definite clause? True


---
# 5.6 Studi Kasus: Knowledge Base Colonel West

## Penjelasan

Skenario:

> The law says that it is a crime for an American to sell weapons to hostile
> nations. The country Nono, an enemy of America, has some missiles, and all of
> its missiles were sold to it by Colonel West, who is American.

Target Notebook 06: membuktikan `Criminal(West)`.

### Langkah 1: vocabulary

| Symbol | Arity | Makna |
|---|---:|---|
| `American(x)` | 1 | x orang Amerika |
| `Weapon(x)` | 1 | x senjata |
| `Missile(x)` | 1 | x rudal |
| `Owns(x,y)` | 2 | x memiliki y |
| `Enemy(x,y)` | 2 | x musuh y |
| `Hostile(x)` | 1 | x hostile |
| `Sells(x,y,z)` | 3 | x menjual y kepada z |
| `Criminal(x)` | 1 | x kriminal |

### Langkah 2: FOL formal dan encoding

| Natural language | FOL formal / encoding executable |
|---|---|
| Orang Amerika yang menjual senjata ke pihak hostile adalah kriminal | $\forall x\forall y\forall z((American(x)\land Weapon(y)\land Sells(x,y,z)\land Hostile(z))\Rightarrow Criminal(x))$ |
| Nono memiliki suatu rudal | Formal: $\exists x(Owns(Nono,x)\land Missile(x))$. Encoding dengan fresh witness: `Owns(Nono,M1)`, `Missile(M1)` |
| Semua rudal milik Nono dijual oleh West kepada Nono | $\forall x((Missile(x)\land Owns(Nono,x))\Rightarrow Sells(West,x,Nono))$ |
| Semua rudal adalah senjata | $\forall x(Missile(x)\Rightarrow Weapon(x))$ |
| Semua musuh America hostile | $\forall x(Enemy(x,America)\Rightarrow Hostile(x))$ |
| West orang Amerika | $American(West)$ |
| Nono musuh America | $Enemy(Nono,America)$ |

Semua universal quantifier pada rule dihilangkan saat di-encode karena
`FolKB` memperlakukannya secara implisit.

## Contoh penerapan


In [ ]:
crime_rule = expr(
    '(American(x) & Weapon(y) & Sells(x, y, z) & Hostile(z)) '
    '==> Criminal(x)'
)
sells_missile = expr(
    '(Missile(x) & Owns(Nono, x)) ==> Sells(West, x, Nono)'
)
missile_is_weapon = expr('Missile(x) ==> Weapon(x)')
enemy_is_hostile = expr('Enemy(x, America) ==> Hostile(x)')

facts = [
    expr('Owns(Nono, M1)'),   # M1 adalah fresh witness.
    expr('Missile(M1)'),
    expr('American(West)'),
    expr('Enemy(Nono, America)'),
]

my_crime_kb = FolKB([
    crime_rule,
    sells_missile,
    missile_is_weapon,
    enemy_is_hostile,
    *facts,
])

for clause in my_crime_kb.clauses:
    print(clause)


((((American(x) & Weapon(y)) & Sells(x, y, z)) & Hostile(z)) ==> Criminal(x))
((Missile(x) & Owns(Nono, x)) ==> Sells(West, x, Nono))
(Missile(x) ==> Weapon(x))
(Enemy(x, America) ==> Hostile(x))
Owns(Nono, M1)
Missile(M1)
American(West)
Enemy(Nono, America)


In [ ]:
# Pastikan KB buatan kita memuat clause yang sama dengan crime_kb bawaan.
my_clauses = {str(clause) for clause in my_crime_kb.clauses}
builtin_clauses = {str(clause) for clause in crime_kb.clauses}

print("Jumlah clause buatan :", len(my_clauses))
print("Jumlah clause bawaan :", len(builtin_clauses))
print("Isi clause sama?     :", my_clauses == builtin_clauses)


Jumlah clause buatan : 8
Jumlah clause bawaan : 8
Isi clause sama?     : True


---
# Latihan Soal

## Soal 1 — Klasifikasi syntax

Untuk setiap ekspresi, tentukan apakah bentuk utamanya **term** atau **formula**.
Jika berupa formula, tentukan apakah atomic atau compound serta apakah statusnya
open formula atau sentence. Sebutkan juga arity tiap function/predicate dan
variable yang masih free.

1. `AdvisorOf(Ani)`
2. `Lecturer(AdvisorOf(Ani))`
3. `Takes(x, AI)`
4. $\forall x(Student(x)\Rightarrow Takes(x,AI))$
5. $\exists x(Student(x)\land Takes(x,y))$

Jelaskan mengapa ekspresi 1 tidak dapat disebut benar atau salah sendirian.


In [ ]:
# Soal 1 — klasifikasi syntax
e1 = expr('AdvisorOf(Ani)')
e2 = expr('Lecturer(AdvisorOf(Ani))')
e3 = expr('Takes(x, AI)')

rows = [
    ("AdvisorOf(Ani)",
     "term (function)",
     "arity 1",
     "-",
     "-",
     sorted(str(v) for v in variables(e1))),
    ("Lecturer(AdvisorOf(Ani))",
     "formula atomik",
     "predicate arity 1",
     "sentence (ground, no free var)",
     "formula",
     sorted(str(v) for v in variables(e2))),
    ("Takes(x, AI)",
     "formula atomik",
     "predicate arity 2",
     "open formula (free: x)",
     "formula",
     sorted(str(v) for v in variables(e3))),
    ("forall x (Student(x) ==> Takes(x, AI))",
     "formula majemuk",
     "predicate arity 1 / 2",
     "sentence (x bound by forall)",
     "formula",
     []),
    ("exists x (Student(x) & Takes(x, y))",
     "formula majemuk",
     "predicate arity 1 / 2",
     "open formula (x bound, y free)",
     "formula",
     ["y"]),
]

print(f"{'Ekspresi':42} {'Bentuk utama':18} {'Arity':22} {'Status':40} {'Jenis':8} Free")
for row in rows:
    print(f"{row[0]:42} {row[1]:18} {row[2]:22} {row[3]:40} {row[4]:8} {row[5]}")

# AdvisorOf(Ani) adalah term: menunjuk object (output dosen wali),
# bukan pernyataan, jadi tidak punya nilai benar/salah sendirian.
print("\n1) AdvisorOf(Ani) = term (hasil function), bukan formula -> tidak bisa benar/salah sendirian.")


Ekspresi                                   Bentuk utama       Arity                  Status                                   Jenis    Free
AdvisorOf(Ani)                             term (function)    arity 1                -                                        -        []
Lecturer(AdvisorOf(Ani))                   formula atomik     predicate arity 1      sentence (ground, no free var)           formula  []
Takes(x, AI)                               formula atomik     predicate arity 2      open formula (free: x)                   formula  ['x']
forall x (Student(x) ==> Takes(x, AI))     formula majemuk    predicate arity 1 / 2  sentence (x bound by forall)             formula  []
exists x (Student(x) & Takes(x, y))        formula majemuk    predicate arity 1 / 2  open formula (x bound, y free)           formula  ['y']

1) AdvisorOf(Ani) = term (hasil function), bukan formula -> tidak bisa benar/salah sendirian.


<details>
<summary>Klik untuk melihat hint</summary>

Mulai dari hasil akhir ekspresi. Function application menghasilkan term;
predicate application menghasilkan atomic formula. Formula menjadi sentence
hanya jika semua variable berada dalam scope quantifier. Pada nomor 5, periksa
`x` dan `y` secara terpisah.

</details>


## Soal 2 — Evaluasi pada model

Gunakan interpretation berikut:

```text
D        = {Ani, Budi, Cici}
Student  = {Ani, Budi}
Diligent = {Budi, Cici}
```

Tentukan benar atau salah. Untuk universal statement yang salah, berikan
counterexample. Untuk existential statement yang benar, berikan witness.

1. $\forall x(Student(x)\Rightarrow Diligent(x))$
2. $\exists x(Student(x)\land Diligent(x))$
3. $\forall x(Student(x)\land Diligent(x))$
4. $\exists x(Student(x)\land\neg Diligent(x))$

Implementasikan pemeriksaan dengan `all()` dan `any()`.


In [ ]:
# Soal 2 — evaluasi pada model
D = {"Ani", "Budi", "Cici"}
Student = {"Ani", "Budi"}
Diligent = {"Budi", "Cici"}

# 1) forall x (Student(x) ==> Diligent(x))
q1 = all((x not in Student) or (x in Diligent) for x in D)
ce1 = sorted(x for x in D if x in Student and x not in Diligent)

# 2) exists x (Student(x) & Diligent(x))
q2 = any((x in Student) and (x in Diligent) for x in D)
w2 = sorted(x for x in D if x in Student and x in Diligent)

# 3) forall x (Student(x) & Diligent(x))
q3 = all((x in Student) and (x in Diligent) for x in D)
ce3 = sorted(x for x in D if not ((x in Student) and (x in Diligent)))

# 4) exists x (Student(x) & ~Diligent(x))
q4 = any((x in Student) and (x not in Diligent) for x in D)
w4 = sorted(x for x in D if x in Student and x not in Diligent)

print("1) forall x (Student(x) ==> Diligent(x)) =", q1, "| counterexample:", ce1)
print("2) exists x (Student(x) & Diligent(x))    =", q2, "| witness:", w2)
print("3) forall x (Student(x) & Diligent(x))    =", q3, "| counterexample:", ce3)
print("4) exists x (Student(x) & ~Diligent(x))   =", q4, "| witness:", w4)


1) forall x (Student(x) ==> Diligent(x)) = False | counterexample: ['Ani']
2) exists x (Student(x) & Diligent(x))    = True | witness: ['Budi']
3) forall x (Student(x) & Diligent(x))    = False | counterexample: ['Ani', 'Cici']
4) exists x (Student(x) & ~Diligent(x))   = True | witness: ['Ani']


<details>
<summary>Klik untuk melihat hint</summary>

Representasikan domain dan extension predicate sebagai `set`. Implikasi
$P\Rightarrow Q$ dapat diperiksa sebagai `(not P) or Q`. Jangan hanya mencetak
Boolean; cari object yang menjadi counterexample atau witness.

</details>


## Soal 3 — Memperbaiki quantifier

Dua formula berikut tidak sesuai dengan maksud kalimatnya.

1. “Semua mahasiswa pintar” ditulis
   $\forall x(Student(x)\land Smart(x))$.
2. “Ada mahasiswa pintar” ditulis
   $\exists x(Student(x)\Rightarrow Smart(x))$.

Untuk masing-masing:

- jelaskan makna formula yang salah;
- berikan model kecil yang memperlihatkan masalahnya; dan
- tulis formula yang benar.


In [ ]:
# Soal 3 — memperbaiki quantifier
# Model: semua mahasiswa pintar, ada non-mahasiswa di domain.
domain = {"Ani", "Budi", "Kursi"}
Student = {"Ani", "Budi"}
Smart = {"Ani", "Budi"}

# 1) Salah: forall x (Student(x) & Smart(x))
#    Menuntut SETIAP object di domain = mahasiswa DAN pintar (termasuk Kursi).
wrong1 = all((x in Student) and (x in Smart) for x in domain)
correct1 = all((x not in Student) or (x in Smart) for x in domain)
print("1) forall (Student & Smart) =", wrong1, "(Kursi bukan mahasiswa -> counterexample)")
print("   benar: forall (Student ==> Smart) =", correct1)

# 2) Salah: exists x (Student(x) ==> Smart(x))
#    Vacuously true karena Kursi bukan mahasiswa, padahal belum tentu ada
#    mahasiswa pintar. Contoh: tidak ada mahasiswa pintar sama sekali.
Smart_none = set()
wrong2 = any((x not in Student) or (x in Smart_none) for x in domain)
correct2 = any((x in Student) and (x in Smart_none) for x in domain)
print("2) exists (Student ==> Smart) =", wrong2, "(witness: Kursi, vacuous)")
print("   benar: exists (Student & Smart) =", correct2)


1) forall (Student & Smart) = False (Kursi bukan mahasiswa -> counterexample)
   benar: forall (Student ==> Smart) = True
2) exists (Student ==> Smart) = True (witness: Kursi, vacuous)
   benar: exists (Student & Smart) = False


<details>
<summary>Klik untuk melihat hint</summary>

Masukkan satu object bukan mahasiswa ke domain. Formula pertama masih menuntut
apa dari object tersebut? Pada formula kedua, apa nilai implication ketika
premise `Student(x)` salah?

</details>


## Soal 4 — Scope dan urutan quantifier

Gunakan predicate `Lecturer(x)`, `Student(y)`, dan `Teaches(x,y)` untuk menulis:

1. “Setiap dosen mengajar sedikitnya satu mahasiswa.”
2. “Ada satu mahasiswa yang diajar oleh setiap dosen.”

Jelaskan perbedaan scope dan urutan quantifier. Buat satu model kecil yang
membuat formula pertama benar tetapi formula kedua salah.


In [ ]:
# Soal 4 — scope dan urutan quantifier
# 1) forall x (Lecturer(x) ==> exists y (Student(y) & Teaches(x, y)))
#    "Setiap dosen mengajar sedikitnya satu mahasiswa." (mahasiswa boleh beda)
# 2) exists y (Student(y) & forall x (Lecturer(x) ==> Teaches(x, y)))
#    "Ada satu mahasiswa yang diajar SETIAP dosen." (mahasiswa harus sama)

print("1) forall x (Lecturer(x) ==> exists y (Student(y) & Teaches(x, y)))")
print("2) exists y (Student(y) & forall x (Lecturer(x) ==> Teaches(x, y)))")

lecturers = {"D1", "D2"}
students = {"M1", "M2"}
teaches = {("D1", "M1"), ("D2", "M2")}  # tiap dosen -> mahasiswa berbeda

f1 = all(
    any((x, y) in teaches for y in students)
    for x in lecturers
)
f2 = any(
    all((x, y) in teaches for x in lecturers)
    for y in students
)

print("Model:", "lecturers =", sorted(lecturers), "students =", sorted(students))
print("Teaches =", sorted(teaches))
print("Formula 1 =", f1, "(D1->M1, D2->M2: tiap dosen punya mahasiswa)")
print("Formula 2 =", f2, "(tidak ada satu mahasiswa yang diajar semua dosen)")


1) forall x (Lecturer(x) ==> exists y (Student(y) & Teaches(x, y)))
2) exists y (Student(y) & forall x (Lecturer(x) ==> Teaches(x, y)))
Model: lecturers = ['D1', 'D2'] students = ['M1', 'M2']
Teaches = [('D1', 'M1'), ('D2', 'M2')]
Formula 1 = True (D1->M1, D2->M2: tiap dosen punya mahasiswa)
Formula 2 = False (tidak ada satu mahasiswa yang diajar semua dosen)


<details>
<summary>Klik untuk melihat hint</summary>

Pada kalimat pertama, mahasiswa boleh berbeda untuk tiap dosen. Pada kalimat
kedua, satu mahasiswa yang sama harus menjadi pasangan semua dosen. Mulai
dengan dua dosen dan dua mahasiswa.

</details>


## Soal 5 — Formula formal dan encoding executable

Terjemahkan dua kalimat berikut:

1. “Semua mahasiswa yang mengambil AI dan lulus Pemrograman boleh mengikuti
   praktikum AI.”
2. “Ada mahasiswa yang mengambil AI.”

Untuk tiap kalimat:

- tulis FOL formal;
- tentukan free dan bound variable;
- jika dapat dijalankan oleh `FolKB`, tulis encoding-nya;
- untuk existential statement, gunakan satu fresh constant dan jelaskan kenapa
  constant tersebut harus fresh.

Bangun `FolKB`, cetak semua clause, lalu periksa setiap clause dengan
`is_definite_clause`.


In [ ]:
# Soal 5 — formula formal dan encoding executable
# 1) "Semua mahasiswa yang mengambil AI dan lulus Pemrograman boleh mengikuti praktikum AI."
#    Formal: forall x ((Student(x) & Takes(x, AI) & Passes(x, Programming))
#                      ==> Eligible(x, AIPracticum))
#    Free/bound: x bound oleh forall; encoding FolKB menganggap x universal implisit.
rule = expr(
    "(Student(x) & Takes(x, AI) & Passes(x, Programming)) "
    "==> Eligible(x, AIPracticum)"
)

# 2) "Ada mahasiswa yang mengambil AI."
#    Formal: exists x (Student(x) & Takes(x, AI))
#    x bound oleh exists. FolKB tidak menyimpan exists -> witness fresh S1.
exist_facts = [
    expr("Student(S1)"),
    expr("Takes(S1, AI)"),
]

kb = FolKB([rule, *exist_facts])

print("1) Formal : forall x ((Student(x) & Takes(x, AI) & Passes(x, Programming)) ==> Eligible(x, AIPracticum))")
print("   Encoding:", rule)
print("   Free vars encoding:", sorted(str(v) for v in variables(rule)), "-> bound implisit oleh FolKB")
print()
print("2) Formal : exists x (Student(x) & Takes(x, AI))")
print("   Encoding witness fresh S1:", exist_facts[0], "&", exist_facts[1])
print("   (S1 harus fresh agar tidak bentrok arti dengan constant lain di KB)")
print()
print("Clauses dalam FolKB:")
for clause in kb.clauses:
    print(" ", clause, "| definite?", is_definite_clause(clause))


1) Formal : forall x ((Student(x) & Takes(x, AI) & Passes(x, Programming)) ==> Eligible(x, AIPracticum))
   Encoding: (((Student(x) & Takes(x, AI)) & Passes(x, Programming)) ==> Eligible(x, AIPracticum))
   Free vars encoding: ['x'] -> bound implisit oleh FolKB

2) Formal : exists x (Student(x) & Takes(x, AI))
   Encoding witness fresh S1: Student(S1) & Takes(S1, AI)
   (S1 harus fresh agar tidak bentrok arti dengan constant lain di KB)

Clauses dalam FolKB:
  (((Student(x) & Takes(x, AI)) & Passes(x, Programming)) ==> Eligible(x, AIPracticum)) | definite? True
  Student(S1) | definite? True
  Takes(S1, AI) | definite? True


<details>
<summary>Klik untuk melihat hint</summary>

Kalimat pertama memiliki tiga premise sebelum `Eligible`. Variable rule menjadi
universal secara implisit dalam `FolKB`. Kalimat kedua perlu ditulis sebagai
$\exists x$ pada tahap formal, lalu diberi nama witness baru pada tahap
encoding.

</details>
